# 05 · Aplicar los modelos a todos los contratos con Spark

Los modelos se entrenaron con 50.000 contratos. Ahora los aplicamos a los **≈5,6 millones de contratos válidos**.

**¿Por qué Spark aquí?** Cada contrato se evalúa por separado, sin depender de los demás. Spark divide los datos en particiones y evalúa cada una en un núcleo distinto (`mapInPandas`). Si tuviéramos un clúster con más máquinas, el mismo código correría más rápido sin cambiar nada.

**Cómo funciona `mapInPandas`:** Spark entrega cada partición como uno o varios DataFrames de pandas a nuestra función, que devuelve el mismo DataFrame con columnas nuevas. Los modelos se envían a todos los núcleos con un **broadcast** (una copia de solo lectura para cada uno).

In [ ]:
from pathlib import Path

RAIZ = Path("..")
DATA = RAIZ / "data"
MUESTRAS = DATA / "muestras"
RESULTADOS = RAIZ / "resultados"
MODELOS = RESULTADOS / "modelos"
FIGURAS = RESULTADOS / "figuras"
TABLAS = RESULTADOS / "tablas"
for carpeta in [MUESTRAS, MODELOS, FIGURAS, TABLAS]:
    carpeta.mkdir(parents=True, exist_ok=True)

NUMERICAS = [
    "log_valor_contrato",          # log(1 + valor del contrato)
    "log_ratio_contrato_proceso",  # log(valor del contrato / valor estimado del proceso)
    "ratio_adiciones",             # valor adicionado / valor del contrato
    "log_plazo_dias",              # log(1 + plazo en días)
    "log_adiciones_dias",          # log(1 + días adicionados)
    "slog_dias_firma_cargue",      # log "con signo" de los días entre la firma y la publicación
]
CATEGORICAS = [
    "modalidad_de_contratacion",
    "tipo_de_contrato",
    "orden_entidad",
    "nombre_grupo",
    "tipo_identifi_del_contratista",
    "destino_gasto",
    "estado_del_proceso",
]

import os
import time
import numpy as np
import pandas as pd
import joblib
from sklearn.neighbors import NearestNeighbors

for variable in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"]:
    os.environ[variable] = "1"
os.environ["OBJC_DISABLE_INITIALIZE_FORK_SAFETY"] = "YES"

import os
import sys
import subprocess

if sys.platform == "darwin":
    os.environ["JAVA_HOME"] = subprocess.check_output(["/usr/libexec/java_home", "-v", "17"]).decode().strip()

os.environ["PYSPARK_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType, BooleanType

spark = (
    SparkSession.builder.appName("secop-poblacion")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.sql.ansi.enabled", "false")
    .config("spark.sql.execution.arrow.maxRecordsPerBatch", "20000")
    .getOrCreate()
)
SILVER = DATA / "silver" / "secop_i_limpio.parquet"
GOLD = DATA / "gold" / "contratos_puntuados.parquet"

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/25 18:12:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1. Cargar los modelos entrenados

In [2]:
prep = joblib.load(MODELOS / "preparacion_limpia.joblib")
kp = joblib.load(MODELOS / "kprototypes_limpio.joblib")
db = joblib.load(MODELOS / "dbscan_limpio.joblib")
N_NUM = len(NUMERICAS)
GAMMA = kp["gamma"]
EPS = db["eps"]
print(f"K-prototypes: {len(kp['centros_num'])} prototipos | DBSCAN: {len(db['Z_nucleos']):,} puntos núcleo, eps = {EPS:.3f}")

K-prototypes: 4 prototipos | DBSCAN: 48,444 puntos núcleo, eps = 1.709


## 2. ¿Cómo aplicar DBSCAN a contratos nuevos?

DBSCAN de scikit-learn no tiene `predict`. Usamos la definición del algoritmo: un contrato nuevo **pertenece a un cluster si tiene algún punto núcleo a distancia ≤ eps**; si no, es **ruido**.

Buscar el núcleo más cercano entre ~40.000 núcleos para 5,6 millones de contratos es lento (en la prueba, más de 1 hora). Por eso usamos un **atajo** que da el mismo resultado:

1. Primero buscamos solo entre los núcleos con **exactamente las mismas categorías** que el contrato. Entre ellos la distancia depende solo de las 6 variables numéricas, y esa búsqueda es muy rápida. Si encontramos uno a ≤ eps, el contrato **no es ruido** y tomamos ese cluster.
2. Solo los contratos que no encontraron núcleo en el paso 1 (casi siempre los que serán ruido) se buscan entre **todos** los núcleos.

El resultado (ruido / no ruido) es el mismo que buscar siempre entre todos los núcleos; solo cambia la velocidad.

In [ ]:
def clave_categorias(X_cat):
    return ["-".join(map(str, fila)) for fila in X_cat]

claves_nucleos = pd.Series(clave_categorias(db["X_cat_nucleos"]))
buscadores = {}
for clave, posiciones in claves_nucleos.groupby(claves_nucleos).groups.items():
    pos = np.asarray(posiciones)
    buscadores[clave] = (NearestNeighbors(n_neighbors=1).fit(db["Z_nucleos"][pos, :N_NUM]),
                         db["cluster_nucleos"][pos])

buscador_general = NearestNeighbors(n_neighbors=1, algorithm="kd_tree").fit(db["Z_nucleos"])
print(f"{len(buscadores):,} combinaciones de categorías distintas entre los núcleos")

4,121 combinaciones de categorías distintas entre los núcleos


## 3. Función que evalúa cada lote de contratos

In [ ]:
modelos = spark.sparkContext.broadcast({
    "prep": prep, "kp": kp, "buscadores": buscadores, "buscador_general": buscador_general,
    "cluster_nucleos": db["cluster_nucleos"],
})

def transformar(lote, m):
    """Las mismas transformaciones del notebook 02."""
    p = m["prep"]
    X_num = p["escalador"].transform(p["imputador"].transform(lote[NUMERICAS]))
    X_cat = p["codificador"].transform(lote[CATEGORICAS].astype(str)).astype(int)
    Z = np.hstack([X_num, p["onehot"].transform(X_cat) * np.sqrt(GAMMA / 2)])
    return X_num, X_cat, Z

def aplicar_kprototypes(X_num, X_cat, m):
    kp = m["kp"]
    parte_num = ((X_num[:, None, :] - kp["centros_num"][None]) ** 2).sum(axis=2)
    parte_cat = (X_cat[:, None, :] != kp["centros_cat"][None]).sum(axis=2)
    D = parte_num + GAMMA * parte_cat
    cluster = D.argmin(axis=1)
    distancia = D.min(axis=1)
    puntaje = distancia / kp["limites"][cluster]
    return cluster, distancia, puntaje

def aplicar_dbscan(X_num, X_cat, Z, m):
    cluster = np.full(len(Z), -1)
    distancia = np.full(len(Z), np.inf)
    claves = pd.Series(clave_categorias(X_cat))
    for clave, filas in claves.groupby(claves).groups.items():
        if clave in m["buscadores"]:
            buscador, etiquetas = m["buscadores"][clave]
            filas = np.asarray(filas)
            d, i = buscador.kneighbors(X_num[filas])
            cerca = d[:, 0] <= EPS
            cluster[filas[cerca]] = etiquetas[i[cerca, 0]]
            distancia[filas[cerca]] = d[cerca, 0]
    pendientes = np.where(cluster == -1)[0]
    if len(pendientes) > 0:
        d, i = m["buscador_general"].kneighbors(Z[pendientes])
        cerca = d[:, 0] <= EPS
        cluster[pendientes[cerca]] = m["cluster_nucleos"][i[cerca, 0]]
        distancia[pendientes] = d[:, 0]
    return cluster, distancia


def evaluar_lotes(lotes):
    m = modelos.value
    for lote in lotes:
        X_num, X_cat, Z = transformar(lote, m)
        kp_cluster, kp_distancia, kp_puntaje = aplicar_kprototypes(X_num, X_cat, m)
        db_cluster, db_distancia = aplicar_dbscan(X_num, X_cat, Z, m)
        lote["kp_cluster"] = kp_cluster.astype("int32")
        lote["kp_distancia"] = kp_distancia.astype("float32")
        lote["kp_puntaje"] = kp_puntaje.astype("float32")
        lote["kp_atipico"] = kp_puntaje > 1
        lote["db_cluster"] = db_cluster.astype("int32")
        lote["db_distancia"] = db_distancia.astype("float32")
        lote["db_ruido"] = db_cluster == -1
        yield lote

## 4. Ejecutar sobre todos los contratos válidos

In [ ]:
contratos = (spark.read.parquet(str(SILVER))
             .filter(F.col("motivo_exclusion").isNull())
             .drop(*["raw_" + c for c in NUMERICAS + CATEGORICAS]))

contratos = contratos.repartition(48)

columnas_nuevas = [
    StructField("kp_cluster", IntegerType()), StructField("kp_distancia", FloatType()),
    StructField("kp_puntaje", FloatType()), StructField("kp_atipico", BooleanType()),
    StructField("db_cluster", IntegerType()), StructField("db_distancia", FloatType()),
    StructField("db_ruido", BooleanType()),
]
esquema_salida = StructType(contratos.schema.fields + columnas_nuevas)

inicio = time.time()
contratos.mapInPandas(evaluar_lotes, schema=esquema_salida).write.mode("overwrite").parquet(str(GOLD))
segundos = time.time() - inicio

gold = spark.read.parquet(str(GOLD))
n = gold.count()
print(f"{n:,} contratos evaluados en {segundos / 60:.1f} minutos ({n / segundos:,.0f} contratos por segundo)")

26/11/25 18:13:06 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


5,628,310 contratos evaluados en 16.2 minutos (5,783 contratos por segundo)


In [6]:
gold.agg(
    F.round(100 * F.avg(F.col("db_ruido").cast("int")), 2).alias("% ruido DBSCAN"),
    F.round(100 * F.avg(F.col("kp_atipico").cast("int")), 2).alias("% atípicos K-prototypes"),
    F.round(100 * F.avg((F.col("db_ruido") & F.col("kp_atipico")).cast("int")), 2).alias("% atípicos en ambos"),
).show()

+--------------+-----------------------+-------------------+
|% ruido DBSCAN|% atípicos K-prototypes|% atípicos en ambos|
+--------------+-----------------------+-------------------+
|          1.47|                   3.03|               0.96|
+--------------+-----------------------+-------------------+



**Verificación:** estos porcentajes son parecidos a los de la muestra de 50.000 (notebooks 03 y 04), entonces la muestra era representativa y entrenar con ella fue razonable.

In [7]:
spark.stop()